# VR Application Observation Data Analysis

This notebook analyzes VR application performance data, focusing on the relationship between number of clients, total stalls, and number of pods.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Load and Explore the Data

In [ ]:
# Load the CSV file
df = pd.read_csv('notebooks/vr_application_observation_table.csv')

# Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# Display basic statistics
print("Basic Statistics for Key Variables:")
df[['num_pods', 'num_clients', 'total_stall']].describe()

## 3. Main Visualization: Total Stall vs Number of Clients with Pods Information

In [ ]:
# Create figure with dual y-axes
fig, ax1 = plt.subplots(figsize=(14, 8))

# Plot total_stall vs num_clients on the first y-axis
color1 = 'tab:blue'
ax1.set_xlabel('Number of Clients', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Stall', fontsize=12, fontweight='bold', color=color1)
ax1.scatter(df['num_clients'], df['total_stall'], alpha=0.6, s=50, color=color1, label='Total Stall')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)

# Create second y-axis for num_pods
ax2 = ax1.twinx()
color2 = 'tab:orange'
ax2.set_ylabel('Number of Pods', fontsize=12, fontweight='bold', color=color2)
ax2.scatter(df['num_clients'], df['num_pods'], alpha=0.6, s=50, color=color2, marker='s', label='Num Pods')
ax2.tick_params(axis='y', labelcolor=color2)

# Add title
plt.title('VR Application Performance: Total Stall and Number of Pods vs Number of Clients', 
          fontsize=14, fontweight='bold', pad=20)

# Add legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Alternative Visualization: Color-coded by Number of Pods

In [ ]:
# Create scatter plot where color represents num_pods
fig, ax = plt.subplots(figsize=(14, 8))

scatter = ax.scatter(df['num_clients'], df['total_stall'], 
                     c=df['num_pods'], cmap='viridis', 
                     s=100, alpha=0.7, edgecolors='black', linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Number of Pods', fontsize=12, fontweight='bold')

# Labels and title
ax.set_xlabel('Number of Clients', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Stall', fontsize=12, fontweight='bold')
ax.set_title('Total Stall vs Number of Clients (Color-coded by Number of Pods)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Grouped Analysis: Average Total Stall by Number of Clients and Pods

In [ ]:
# Group data by num_clients and num_pods to see average total_stall
grouped_data = df.groupby(['num_clients', 'num_pods'])['total_stall'].mean().reset_index()

# Pivot for heatmap
pivot_data = grouped_data.pivot(index='num_pods', columns='num_clients', values='total_stall')

# Create heatmap
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='YlOrRd', 
            cbar_kws={'label': 'Average Total Stall'}, ax=ax)

ax.set_xlabel('Number of Clients', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Pods', fontsize=12, fontweight='bold')
ax.set_title('Average Total Stall by Number of Clients and Pods', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## 6. Line Plot: Total Stall Trend by Number of Pods

In [ ]:
# Create line plot for each pod configuration
fig, ax = plt.subplots(figsize=(14, 8))

# Get unique pod values
unique_pods = sorted(df['num_pods'].unique())

for pod_count in unique_pods:
    pod_data = df[df['num_pods'] == pod_count].groupby('num_clients')['total_stall'].mean().reset_index()
    ax.plot(pod_data['num_clients'], pod_data['total_stall'], 
            marker='o', linewidth=2, markersize=8, label=f'{pod_count} Pod(s)')

ax.set_xlabel('Number of Clients', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Total Stall', fontsize=12, fontweight='bold')
ax.set_title('Average Total Stall Trend by Number of Clients (Grouped by Pods)', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(title='Number of Pods', fontsize=10, title_fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Statistical Summary

In [ ]:
# Correlation analysis
print("Correlation Matrix:")
correlation = df[['num_pods', 'num_clients', 'total_stall']].corr()
print(correlation)
print("\n" + "="*50)

# Summary by number of clients
print("\nAverage Total Stall by Number of Clients:")
client_summary = df.groupby('num_clients')['total_stall'].agg(['mean', 'std', 'min', 'max', 'count'])
print(client_summary)
print("\n" + "="*50)

# Summary by number of pods
print("\nAverage Total Stall by Number of Pods:")
pod_summary = df.groupby('num_pods')['total_stall'].agg(['mean', 'std', 'min', 'max', 'count'])
print(pod_summary)